In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ec4yragw/unsloth_8c9c9c7a520b446f8e890032a381a3fc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ec4yragw/unsloth_8c9c9c7a520b446f8e890032a381a3fc
  Resolved https://github.com/unslothai/unsloth.git to commit 35f887d7956bfc0d0fb35dba9c8ce0623d7d2243
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00
  

In [1]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch

max_seq_length = 2048

tokenizer = AutoTokenizer.from_pretrained("gmz1234/bpe")

model, _ = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit = False,
    token = HF_TOKEN
)

model.resize_token_embeddings(len(tokenizer))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Embedding(1024, 2560)

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)

Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.7.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM


In [5]:
from datasets import load_dataset

CUSTOM_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n' }}"
    "{% endfor %}"
)
dataset = load_dataset("gmz1234/stackoverflow_ai", split="train", token=HF_TOKEN)

def formatting_prompts_func(examples):
    conversations = examples["messages"]
    output_texts = []

    for message_list in conversations:
        if isinstance(message_list, dict):
            message_list = [message_list]

        formatted_messages = []
        for msg in message_list:
            formatted_messages.append({
                "role": msg["role"],
                "content": msg["context"]
            })

        text = tokenizer.apply_chat_template(
            formatted_messages,
            chat_template = CUSTOM_CHAT_TEMPLATE,
            tokenize = False,
            add_generation_prompt = False
        )
        output_texts.append(text)

    return { "text" : output_texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
dataset

Dataset({
    features: ['conversation_id', 'messages', 'text'],
    num_rows: 1000
})

In [11]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

training_args = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 10,
    max_steps = 120,
    learning_rate = 2e-4,
    fp16 = True,
    bf16 = False,
    logging_steps = 1,
    output_dir = "qwen3_fine_tune_results",
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 38,123,520 of 3,676,878,336 (1.04% trained)


Step,Training Loss
1,4.266298
2,4.553729
3,4.911182
4,4.583252
5,4.349959
6,4.691195
7,4.935581
8,4.437239
9,4.575355
10,4.662235


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


In [12]:
# Kalıcı olması için chat şablonunu tokenizer içine gömüyoruz
tokenizer.chat_template = CUSTOM_CHAT_TEMPLATE

# Modeli ve tokenizer'ı yerel klasöre kaydetme
model.save_pretrained("qwen3_custom_lora_model")
tokenizer.save_pretrained("qwen3_custom_lora_model")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


('qwen3_custom_lora_model/tokenizer_config.json',
 'qwen3_custom_lora_model/chat_template.jinja',
 'qwen3_custom_lora_model/tokenizer.json')

In [24]:
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=True)

In [25]:
repo_id = "gmz1234/qwen3-custom-lora"

model.push_to_hub(repo_id, token = True)

tokenizer.push_to_hub(repo_id, token = True)

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  569kB /  158MB            

Saved model to https://huggingface.co/gmz1234/qwen3-custom-lora


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gmz1234/qwen3-custom-lora/commit/feea5f53e5d34a757265726ce6363bbc56cad56c', commit_message='Upload tokenizer', commit_description='', oid='feea5f53e5d34a757265726ce6363bbc56cad56c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gmz1234/qwen3-custom-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='gmz1234/qwen3-custom-lora'), pr_revision=None, pr_num=None)